## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [1]:
# Import Required Libraries.
import os
import sys

import json
import subprocess

import pandas as pd
from pathlib import Path

from glob import glob
from logging_module import LogLevel, log

# Set Working Directory to Project Root.
os.chdir("..") 

# Add Project Root to Path.
sys.path.insert(0, os.getcwd())


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Import Data Downloader Libraries.
from src.data.download_data import main as downloadData

# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "data/hard_dataset_flights.txt"

# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)

In [6]:
# Set Dataset Ground Truth Folders.
DATASET_PARTS = [
    "data/part1/ImageSets",
    "data/part2/ImageSets",
    "data/part3/ImageSets",
]

# Set Corresponding Inference Image Folders.
IMAGE_PARTS = [
    "data/part1/Images",
    "data/part2/Images",
    "data/part3/Images",
]

# Set Output Results Folders.
# TODO: Confirm these match DATASET_ENV values used during inference.
RESULTS_DIRS = [
    "data/results/baseline/part1",
    "data/results/baseline/part2",
    "data/results/baseline/part3",
]

# Aggregated Summaries Output Folder.
SUMMARIES_DIR = "data/results/baseline/summaries"

# Path to the Metrics Runner Script.
METRICS_SCRIPT = "core/metrics/run_airborne_metrics.py"

# Minimum Score Threshold.
MIN_SCORE = 0.985

# Minimum Track Length.
MIN_TRACK_LEN = 30

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Parts : {len(DATASET_PARTS)}")
print(f"  Results Dirs  : {RESULTS_DIRS}")
print(f"  Summaries Dir : {SUMMARIES_DIR}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

<INFO ><baseline:config> Configuration Loaded.
  Dataset Parts : 3
  Results Dirs  : ['data/results/baseline/part1', 'data/results/baseline/part2', 'data/results/baseline/part3']
  Summaries Dir : data/results/baseline/summaries
  Min Score     : 0.985
  Min Track Len : 30



---

<br/>